## Test Day with pretty ugly data

In [ ]:
if False:
    import os
    os.environ["QUANTDATA_CACHE_MODE"] = "off" # turns off global cache mode; good for testing, but do not use in general.

%load_ext autoreload
%autoreload 1

%aimport quantdata
%aimport quantdata.biz_calendar
%aimport quantdata.utils.arbfree
%aimport quantdata.widb
%aimport quantdata.widb.client
%aimport quantdata.widb.day
%aimport quantdata.widb.arbfree

from datetime import date

from cdxcore.dynaplot import figure
from quantdata.widb.day import collect_raw_day, enrich_day, ExpiryFilter, RawDayFilter
from quantdata.widb.arbfree import mk_arb_free_day, MkArbFree
from quantdata.biz_calendar import next_business_day, DAYS_PER_YEAR
from cdxcore.verbose import Context
from IPython.display import display
import numpy as np

def test():
    ticker = "SPX"
    d      = next_business_day( date(2012, 3, 28), count_start=True )

    raw_day_filter = RawDayFilter()
    expiry_filter = ExpiryFilter()
    verbose = Context.all

    raw_day = collect_raw_day(ticker, d, raw_day_filter=raw_day_filter, verbose=verbose, raise_if_bad=False)

    enr_day = enrich_day(ticker, d, raw_day, raw_day_filter=raw_day_filter, expiry_filter=expiry_filter, verbose=verbose, raise_if_bad=False)

    expiry_ids = enr_day['expiry_id'].to_numpy()
    unids = np.unique( expiry_ids )

    with figure("Expiries", row_size=3, col_size=4, columns=4) as fig:
        for uid in unids:
            tte_y = enr_day['tte_y'][ expiry_ids == uid ].iloc[0]

            mask = enr_day['expiry_id'] == uid
            strikes = enr_day['pure_strike'][mask].to_numpy()
            bids = enr_day['pure_bid_price'][mask].to_numpy()
            asks = enr_day['pure_ask_price'][mask].to_numpy()
            mids = (bids + asks) / 2.

            dC = np.diff( mids, prepend=1. ) / np.diff( strikes, prepend=0. )
            p  = np.diff( mids, append=0.) 

            ok = np.all(p>=-1E-6) and abs(np.sum(p)-1) < 1E-6 and abs(p@strikes-1.) < 1E-6
            ax = fig.add_subplot(f"{uid} {tte_y*DAYS_PER_YEAR:.1f}d {'ok' if ok else 'not ok'}")
            ax.plot(strikes, dC, label="market dC")
            ax.plot(strikes, strikes*0-1., ":", color="black")
            ax.plot(strikes, strikes*0, ":", color="black")
            ax.legend(loc="upper left")
            ax2 = ax.twinx()
            ax2.plot(strikes, p, label="market p", color="orange")
            ax.plot(strikes, strikes*0-1., ":", color="red")
            ax.plot(strikes, strikes*0, ":", color="red")
            ax2.legend(loc="upper right")

    """
    mask = enr_day['expiry_id'] == BAD_UID
    tte_y = enr_day['tte_y'][ mask].iloc[0]
    expiry = enr_day['expiry'][ mask].iloc[0]
    is_am = enr_day['is_am'][ mask].iloc[0]
    strikes = enr_day['pure_strike'][mask].to_numpy()
    bids = enr_day['pure_bid_price'][mask].to_numpy()
    asks = enr_day['pure_ask_price'][mask].to_numpy()
    mids = (bids + asks) / 2.
    intr = np.maximum( 0., 1. - strikes )
    
    with figure(f"{expiry} {'am' if is_am else 'pm'}", row_size=10, col_size=14, columns=1) as fig:
        ax = fig.add_subplot()

        ax.plot(strikes, mids-intr, ".-", color="orange", label="mid")
        ax.plot(strikes, bids-intr,"^:", color="green", label="bid")
        ax.plot(strikes, asks-intr, "v:", color="red", label="ask")
        ax.legend(loc="upper left")
    """


    arb_free = MkArbFree(timeout_seconds_expiry=500)
    arb_free = mk_arb_free_day(ticker, d, enr_day, mk_arb_free=arb_free, raw_day_filter=raw_day_filter, expiry_filter=expiry_filter, verbose=verbose, raise_if_bad=False, override_cache_mode="off")

    display(arb_free)
    return arb_free

_ = test()

00: SPX 2012-03-28: Enriched 454 of 884 raw options across 6 of 6 input expiries.
01:   Dropped the following 8 of 14 input expiries: 2012-06-29pm (insufficient traded strikes), 2012-07-21am (insufficient traded strikes), 2012-09-28pm (insufficient traded strikes), 2012-12-31pm (insufficient traded strikes), 2013-03-16am (insufficient traded strikes), 2013-06-22am (insufficient traded strikes), 2013-12-21am (insufficient traded strikes) and 2014-12-20am (insufficient traded strikes). 
01:   Dropped 62 strikes among the remaining 6 expiries: 2012-03-30pm (Dropped 7 strikes with p/c issues: 26/33 strikes left), 2012-04-21am (Dropped 17 strikes with p/c issues: 82/99 strikes left), 2012-05-19am (Dropped 13 strikes with p/c issues: 115/128 strikes left), 2012-06-16am (Dropped 17 strikes with p/c issues: 132/149 strikes left), 2012-09-22am (Dropped 3 strikes with p/c issues: 45/48 strikes left) and 2012-12-22am (Dropped 5 strikes with p/c issues: 54/59 strikes left). 


IndexError: single positional indexer is out-of-bounds